# 04. バックキャスティング（Backcasting） — 練習問題

**対象技術**: 量子コンピューティング（耐量子計算暗号 PQC への移行）

望ましい未来（ゴール）をまず定義し、そこから現在へ逆算してマイルストーンと施策を導く規範的手法である。フォアキャスティング（予測の延長）の逆向きにあたる。本ノートブックでは PQC 移行のロードマップを S 字曲線で逆算配置し、遅延時のリカバリ計画を定量化する。

`numpy` でロジスティック曲線を計算し、`matplotlib` で移行ロードマップを可視化する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## ゴールと現状の定義

「2035年までに重要インフラの暗号 100% を PQC へ移行」をゴールに設定し、逆算配置の検証に使うマイルストーンを年ごとに定義する。

In [ ]:
CURRENT_YEAR = 2026
GOAL_YEAR = 2035
CURRENT_RATE = 5.0    # 現在の PQC 移行率 [%]
GOAL_RATE = 100.0     # ゴールの移行率 [%]

# マイルストーン: 年 -> (ラベル, 必要累積移行率の目安 [%])
MILESTONES = {
    2026: ("暗号資産の棚卸完了", 5),
    2027: ("PQC 調達義務化", 8),
    2029: ("PQC 準拠製品の本格出荷", 20),
    2031: ("政府システム移行完了", 60),
    2033: ("金融セクター移行完了", 85),
    2035: ("全重要インフラ移行完了", 100),
}

print(f"ゴール: {GOAL_YEAR}年までに PQC 移行率 {GOAL_RATE:.0f}%")
print(f"現状 ({CURRENT_YEAR}年): {CURRENT_RATE:.0f}%  / ギャップ "
      f"{GOAL_RATE - CURRENT_RATE:.0f} ポイント")

## 解析関数

S 字（ロジスティック）曲線で必要累積移行率を逆算配置する関数と、残り年数での年間必要ペースを計算する関数を定義する。

In [ ]:
def logistic_curve(years, y0, y1, k=0.9):
    """ゴール年に y1、現在年に y0 へ到達するS字(ロジスティック)曲線。

    移行は初期緩慢・中期加速・終盤飽和。線形配置を避けるため曲線で配置する。
    """
    mid = (years[0] + years[-1]) / 2.0
    raw = 1.0 / (1.0 + np.exp(-k * (years - mid)))
    r0, r1 = raw[0], raw[-1]
    return y0 + (raw - r0) / (r1 - r0) * (y1 - y0)


def required_pace(target_rate, current_rate, years_left):
    """残り年数で target に届くための年間必要ペース [ポイント/年]。"""
    if years_left <= 0:
        return float("inf")
    return (target_rate - current_rate) / years_left

## S字曲線で必要累積移行率を逆算配置

現在年からゴール年までの各年について、必要累積移行率を S 字配置で逆算する。

In [ ]:
years = np.arange(CURRENT_YEAR, GOAL_YEAR + 1)
plan = logistic_curve(years.astype(float), CURRENT_RATE, GOAL_RATE)

print("逆算された各年の必要累積移行率(S字配置):")
print(f"  {'年':>6} | {'必要移行率':>10} | {'年間増分':>9} | マイルストーン")
print("  " + "-" * 58)
prev = CURRENT_RATE
for i, yr in enumerate(years):
    rate = plan[i]
    delta = rate - prev
    ms = MILESTONES.get(int(yr), ("", None))[0]
    print(f"  {int(yr):>6} | {rate:>9.1f}% | {delta:>8.1f} | {ms}")
    prev = rate

## マイルストーン整合チェック

各マイルストーンの目安値と S 字配置の差が許容範囲内かを確認する。

In [ ]:
print("マイルストーン目安と S字配置の整合チェック:")
for yr, (label, target) in MILESTONES.items():
    idx = int(yr - CURRENT_YEAR)
    planned = plan[idx]
    gap = planned - target
    flag = "OK" if abs(gap) <= 12 else "要調整"
    print(f"  {yr} {label:<22} 目安{target:>4}% / S字{planned:>6.1f}% "
          f"[{flag}]")

## 遅延シナリオとリカバリ計画

2030年の中間レビューで遅延が発覚したと想定し、残期間でゴールに到達するための必要ペース増を計算する。

In [ ]:
review_year = 2030
planned_at_review = plan[review_year - CURRENT_YEAR]
actual_at_review = 25.0   # 実績(遅延発生)
print(f"■ 遅延シナリオ: {review_year}年の中間レビュー")
print(f"  計画上の累積移行率 : {planned_at_review:.1f}%")
print(f"  実績の累積移行率   : {actual_at_review:.1f}%")
print(f"  遅延 : {planned_at_review - actual_at_review:.1f} ポイント遅れ")
print()

years_left = GOAL_YEAR - review_year
orig_pace = required_pace(GOAL_RATE, planned_at_review, years_left)
recovery_pace = required_pace(GOAL_RATE, actual_at_review, years_left)
print("  リカバリ計画(残期間でゴール100%に到達するための必要ペース):")
print(f"    当初想定ペース : {orig_pace:.1f} ポイント/年")
print(f"    リカバリ必要ペース : {recovery_pace:.1f} ポイント/年")
print(f"    ペース増 : x{recovery_pace / orig_pace:.2f} "
      f"(+{recovery_pace - orig_pace:.1f} ポイント/年)")
print()

capacity_limit = 18.0  # 想定される年間移行能力の上限 [ポイント/年]
if recovery_pace > capacity_limit:
    print(f"  [警告] リカバリ必要ペース {recovery_pace:.1f} は想定能力上限 "
          f"{capacity_limit:.0f} を超過。")
    print("         ゴール年の後ろ倒し、または対象範囲の優先順位づけが必要。")
else:
    print(f"  [判定] リカバリ必要ペースは想定能力上限 {capacity_limit:.0f} "
          "以内。挽回可能。")

## 可視化: 移行ロードマップ

累積移行率の S 字曲線を描き、各マイルストーンを点と注記でプロットする。さらに2030年の遅延シナリオから残期間を直線で挽回するリカバリ経路を別線で重ねる。

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

# S字計画線
ax.plot(years, plan, "o-", color="#3a6ea5", lw=2.4, markersize=6,
        label="planned S-curve")

# マイルストーン(目安値)を点と注記でプロット
for yr, (label, target) in MILESTONES.items():
    ax.scatter(yr, target, s=140, color="#f4a259", edgecolors="#5a5a5a",
               zorder=5)
    ax.annotate(f"{yr}\n{target}%", (yr, target),
                textcoords="offset points", xytext=(8, -22),
                fontsize=8, color="#7a4a10")

# 遅延シナリオの実績点
ax.scatter(review_year, actual_at_review, s=170, color="#d1495b",
           edgecolors="#5a5a5a", zorder=6, marker="X",
           label="actual at 2030 review (delayed)")
ax.annotate(f"delay {planned_at_review - actual_at_review:.0f} pts",
            (review_year, actual_at_review),
            textcoords="offset points", xytext=(10, -4),
            fontsize=9, color="#d1495b")

# リカバリ経路(残期間を直線で挽回)
recovery_years = np.arange(review_year, GOAL_YEAR + 1)
recovery_line = actual_at_review + recovery_pace * (recovery_years - review_year)
ax.plot(recovery_years, recovery_line, "--", color="#d1495b", lw=2.2,
        label=f"recovery path ({recovery_pace:.1f} pts/yr)")

# 想定能力上限の参考線(レビュー以降を上限ペースで進めた場合)
cap_line = actual_at_review + capacity_limit * (recovery_years - review_year)
ax.plot(recovery_years, np.minimum(cap_line, 100), ":", color="#6a994e",
        lw=1.8, label=f"capacity limit ({capacity_limit:.0f} pts/yr)")

ax.axhline(GOAL_RATE, color="#999999", lw=1, linestyle="-.")
ax.set_title("Backcasting: PQC migration roadmap for critical infrastructure",
             fontsize=12)
ax.set_xlabel("year")
ax.set_ylabel("cumulative PQC migration rate [%]")
ax.set_xticks(years)
ax.set_ylim(0, 108)
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

バックキャスティングは、未来デザイン論文において、まず望ましい未来の状態を規範的に定義し、そこから現在へと逆算してマイルストーンを配置し、ゴールへ至る政策経路を描き出す装置として用いられる。論文は典型的に、目標とする到達点を明示し、それを実現するために必要な中間段階を時間軸上に逆向きに割り付け、各段階で誰が何をすべきかという施策の連なりを提示する。結論として読者に手渡されるのは、現状の延長線上の予測ではなく、規範的ゴールへ至るための行動計画である。

この手法がもたらす結論の型は「規範ゴールへの経路」であり、その根底には未来をめぐる独特の時間観がある。バックキャスティングにおいて未来は「予測対象」ではなく「選択対象」として扱われる——どうなるかを当てるのではなく、どうしたいかを定め、そこへ至る道筋を設計する。境界設定の経路で見れば、結論に現れる施策は逆算の射程に入れた行為主体と手段の範囲に限られ、ステアリングの対象外と見なされた力は経路から脱落する。価値の所在はきわめて明示的で、望ましい未来の定義そのものに研究者の価値判断が正面から埋め込まれる。

同時に、この明示性は固有のバイアスと表裏一体である。望ましい未来を研究者が選んだ瞬間に、結論がたどり着く方向はすでに先取りされており、論文は「その未来は良い」という前提のもとで経路の精緻化に向かう。さらにこの手法は、適切な施策を打てばゴールへ系を操舵できるというステアリング可能性への楽観に傾きやすく、外生的な制約や経路依存性、他主体の抵抗を経路設計が過小評価する危険がある。それゆえ、ゴール選択の正当化——なぜその未来が望ましいのか、誰の価値基準によるのか——をどれだけ誠実に論じるかが、この手法を採る論文の生命線となる。

## 発展課題

**課題A**: 年間移行能力の上限（例: 年18ポイントまで）を制約として与え、上限を超えない範囲で貪欲法によりマイルストーンを再スケジュールせよ。上限に阻まれてゴール年に間に合うかを判定する。

**課題B**: 複数のゴール年（2033 / 2035 / 2038）で同じ逆算を行い、必要な年間ペース（平均/最大）がどう変わるかを比較表にまとめよ。